# 🚀 LangGraph Fundamentals: Building Your First AI Agent

## Welcome to the World of Agentic AI! 🤖

This comprehensive notebook will guide you through the fundamentals of **LangGraph** - a cutting-edge framework for creating stateful, intelligent AI agents. You'll learn how to build agents that can think, reason, and make decisions through multi-step workflows.

### 🎯 What You'll Learn

By the end of this tutorial, you'll master:

1. **🔧 Environment Setup** - Configure your development environment with all necessary libraries
2. **🧠 StateGraph Concepts** - Understand nodes, edges, and state management
3. **⚡ Building Basic Agents** - Create your first functional AI agent
4. **🔀 Conditional Logic** - Implement decision-making and routing in your agents
5. **🎨 Visualization** - Generate beautiful diagrams of your agent workflows
6. **🧪 Testing & Debugging** - Validate your agents with comprehensive testing

### 🌟 Why LangGraph?

LangGraph revolutionizes AI development by providing:
- **🔄 Stateful Operations**: Maintain context across multiple interactions
- **📊 Visual Workflows**: See your agent's decision-making process
- **🎯 Conditional Routing**: Create intelligent, adaptive behaviors
- **🛠️ Production Ready**: Built for scalable, real-world applications

### 📋 Prerequisites

- **Python 3.8+** installed on your system
- **Basic Python knowledge** (functions, classes, dictionaries)
- **Azure OpenAI access** (API key and endpoint)
- **Curiosity and enthusiasm** for AI development! 🚀

---

*💡 **Pro Tip**: This notebook is designed to be interactive - run each cell as you go and experiment with the code!*

## 1. 🛠️ Environment Setup & Installation

### ? Package Installation

First, let's install all the required packages for our LangGraph journey. We'll need several key libraries:

- **`langgraph`**: The core framework for building our agent workflows
- **`langchain`**: Foundation library for LLM applications  
- **`langchain-openai`**: Integration with OpenAI/Azure OpenAI models
- **`pydantic`**: Data validation and serialization
- **`grandalf`**: Graph visualization capabilities

### ? What Each Package Does:
- **LangGraph**: Provides StateGraph, nodes, edges, and workflow management
- **LangChain**: Offers message types, schema definitions, and LLM abstractions
- **Azure OpenAI**: Connects to powerful GPT models and embeddings
- **Grandalf**: Enables beautiful graph visualizations of our agent flows

In [ ]:
# Install required packages
# %pip install --quiet pydantic
# %pip install --quiet langgraph langchain langchain-openai openai
# %pip install --quiet langchain-community python-dotenv
# %pip install --quiet grandalf

print("✅ All packages installed successfully!")

## 2. 📚 Library Imports & Core Concepts

### Essential Imports
Now let's import all the necessary libraries for our LangGraph agent development.

### 🔑 Key Components:
- **`StateGraph`**: The main class for creating stateful agent workflows
- **`TypedDict`**: Defines the structure of our agent's state
- **`Message Types`**: HumanMessage, AIMessage, SystemMessage for conversations
- **`Azure OpenAI Classes`**: For connecting to powerful language models

In [ ]:
# Import required libraries for LangGraph agent
import sys
import random
from typing import TypedDict, Literal

# LangGraph and LangChain imports
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from typing import TypedDict
print("✅ All libraries imported successfully!")

## 3. 🔐 Azure OpenAI Configuration

### Setting Up Your AI Brain 🧠

This section configures the connection to Azure OpenAI, which will power our intelligent agents.

### 🔧 Configuration Steps:
1. **Load Credentials**: Import API keys and endpoints securely
2. **Initialize Models**: Set up both chat and embedding models
3. **Error Handling**: Ensure all credentials are properly configured
4. **Validation**: Verify successful connection to Azure OpenAI services

### 💡 Models We're Using:
- **`gpt-4o-mini`**: Fast, efficient model for conversations and reasoning
- **`text-embedding-3-small`**: Creates vector representations of text for similarity search

In [ ]:
sys.path.insert(1, '../')

In [ ]:
import init_creds as creds

AZURE_OPENAI_KEY = creds.get_api_key()
AZURE_OPENAI_ENDPOINT = creds.get_endpoint()
AZURE_OPENAI_API_VERSION = "2025-04-01-preview"

if not AZURE_OPENAI_KEY:
    raise ValueError("No AZURE_OPENAI_KEY set for Azure OpenAI API")
if not AZURE_OPENAI_ENDPOINT:
    raise ValueError("No AZURE_OPENAI_ENDPOINT set for Azure OpenAI API")

llm = AzureChatOpenAI(
    model="gpt-4o-mini",
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)

embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=AZURE_OPENAI_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION
    )

print("✅ LLM and Embeddings initialized successfully!")
print(f"📝 Language Model: {llm.model_name}")
print(f"🔍 Embeddings Model: text-embedding-3-small")

## 4. 🔧 Utility Functions

### Creating Helper Functions
Let's create some utility functions that we'll use in our agent workflows.

In [ ]:
def token_counter(input):
    """
    Simple token counting function that counts words in text.
    
    Args:
        input: Text string to count tokens/words in
    
    Returns:
        int: Number of tokens (words) in the input
    """
    tokens = str(input).split()  # Split the text into words
    count_no = len(tokens)  # Count the number of words
    print(f"🔢 Counted {count_no} tokens in the text")
    return count_no

print("✅ Token counter utility function created!")

## 5. 📊 Understanding StateGraph: The Foundation

### What is State? 🤔
In LangGraph, **state** is the shared memory that flows through your agent workflow. Think of it as a clipboard that each node can read from and write to.

### Why StateGraph?
- **🧠 Memory**: Maintains context across multiple steps
- **🔄 Flow Control**: Enables complex decision-making workflows  
- **📈 Scalability**: Handles simple to complex agent architectures
- **🎯 Type Safety**: Uses TypedDict for clear data contracts

In [ ]:
# Define the state schema for our first agent
class GraphState(TypedDict):
    """
    State schema for our simple LangGraph agent workflow.
    
    This defines the data structure that flows through our graph nodes:
    - user_input: The original question or input from the user
    - llm_response: The AI model's response to the user input
    - token_count: Number of tokens/words in the response (for analytics)
    """
    user_input: str
    llm_response: str
    token_count: int

print("✅ Graph state schema defined!")
print("📋 State includes: user_input, llm_response, token_count")

In [ ]:
# The LLM and token_counter need to be wrapped to work with GraphState properly

def llm_node(state: GraphState) -> GraphState:
    """
    Wrapper function for the LLM that works with GraphState.
    Takes user_input from state and generates llm_response.
    """
    user_input = state.get("user_input", "")
    print(f"🤖 LLM processing: {user_input}")
    
    # Create messages for the LLM
    messages = [
        SystemMessage(content="You are a helpful AI assistant. Provide clear and concise answers to user questions."),
        HumanMessage(content=user_input)
    ]
    
    # Get response from LLM
    response = llm.invoke(messages)
    
    # Return updated state
    return {
        **state,
        "llm_response": response.content
    }

def token_counter_node(state: GraphState) -> GraphState:
    """
    Wrapper function for token counter that works with GraphState.
    Takes llm_response from state and adds token_count.
    """
    llm_response = state.get("llm_response", "")
    
    # Count tokens using our existing function
    count = token_counter(llm_response)
    
    # Return updated state
    return {
        **state,
        "token_count": count
    }

print("✅ Node wrapper functions created!")

## 6. 🏗️ Building Your First Agent Workflow

### The Simple Two-Node Agent
Let's create our first agent with two nodes:
1. **LLM Node**: Processes user input and generates responses
2. **Token Counter Node**: Analyzes the response for metrics

### 🔄 Workflow Design:
```
User Input → LLM Model → Token Counter → End
```

This demonstrates the core concept of sequential processing in LangGraph.

In [ ]:
# Create the workflow using StateGraph
workflow = StateGraph(GraphState)

print("🏗️ Building the corrected agent workflow...")

# Add nodes with proper wrapper functions
workflow.add_node('LLM_Model', llm_node)  # Fixed LLM node
workflow.add_node('Get_Token_Counter', token_counter_node)  # Fixed token counter node

print("✅ Added LLM_Model and Get_Token_Counter nodes")

# Define the flow: START → LLM → Token Counter → END
workflow.add_edge(START, 'LLM_Model')  # Start with LLM
workflow.add_edge('LLM_Model', 'Get_Token_Counter')  # LLM → Token Counter
workflow.add_edge('Get_Token_Counter', END)  # Token Counter → End

print("🔗 Connected: START → LLM_Model → Get_Token_Counter → END")

# Compile the graph
app = workflow.compile()

print("\n✅ Graph compiled successfully!")
print("📊 Final workflow: START → LLM_Model → Get_Token_Counter → END")
print("🎉 Your first AI agent is ready to use!")

## 6.5. 🧪 Testing Your First Agent: Simple Example

### Let's Test the Graph We Just Built! 🚀
Now that we've created our workflow, let's see it in action with a simple test. But first, we need to make small adjustments to ensure our nodes work properly with the StateGraph.

In [ ]:
# Simple test function for our agent
def test_agent(user_question: str):
    """
    Simple test function for our LLM + Token Counter agent.
    
    Args:
        user_question: Question to ask the AI agent
    """
    print(f"🧪 Testing Agent with: '{user_question}'")
    print("=" * 50)
    
    # Create initial state
    initial_state = {
        "user_input": user_question,
        "llm_response": "",
        "token_count": 0
    }
    
    # Run the agent
    result = app.invoke(initial_state)
    
    # Display results
    print(f"\n📝 User Input: {result['user_input']}")
    print(f"🤖 LLM Response: {result['llm_response']}")
    print(f"🔢 Token Count: {result['token_count']}")
    print("=" * 50)
    
    return result

# Test the agent with different questions
print("🚀 Running simple tests...\n")

# Test 1: Simple question
test_agent("What is Python?")

print("\n")

# Test 2: More complex question  
test_agent("Explain machine learning in simple terms.")

print("\n")

# Test 3: Creative request
test_agent("Tell me a joke about programming.")

print("✅ All tests completed!")

## 7. 🎨 Visualizing Your Agent

### See Your Agent in Action! 👁️
Let's generate a beautiful diagram of our agent workflow to understand how data flows through the nodes.

In [ ]:
# Generate and display the graph visualization
from IPython.display import Image, display

print("🎨 Generating visual representation of your agent workflow...")

# Create a beautiful diagram showing the flow
display(Image(app.get_graph().draw_mermaid_png()))

print("✅ Agent workflow diagram generated!")
print("📊 This shows how data flows through your agent nodes")

## 8. 🧠 Advanced Concepts: Conditional Logic & Smart Routing

### Beyond Linear Workflows 🔀
Now let's explore the real power of LangGraph - creating agents that can make decisions and take different paths based on conditions!

### 🎯 What We'll Build:
A smart agent that:
- Starts at a common processing node
- Makes a decision based on some logic
- Routes to different specialized nodes
- Demonstrates conditional workflows

### 🔑 Key Concepts:
- **Conditional Edges**: Dynamic routing based on state or logic
- **Decision Functions**: Code that determines which path to take
- **Parallel Paths**: Multiple possible outcomes from a single decision point

In [ ]:
# Define state for our conditional workflow example
class State(TypedDict):
    """
    Simplified state for demonstrating conditional logic in LangGraph.
    
    This example uses a single string field that gets built up through
    different processing paths, showing how state flows through 
    conditional branches.
    """
    graph_state: str

print("✅ Conditional workflow state schema defined!")
print("📝 State contains: graph_state (string that builds up through nodes)")

In [ ]:
# Define processing nodes for our conditional workflow
def node_1(state):
    """
    Initial processing node - everyone starts here.
    This is where common processing happens before routing decisions.
    """
    print('🔄 Node1: Initial processing...')
    # Append to the graph_state to show the processing flow
    return {'graph_state': state['graph_state'] + " We are"}

def node_2(state):
    """
    Specialized processing path A.
    This node handles one type of workflow branch.
    """
    print('⚡ Node2: Path A processing...')
    return {'graph_state': state['graph_state'] + " building"}

def node_3(state):
    """
    Specialized processing path B.
    This node handles an alternative workflow branch.
    """
    print('🚀 Node3: Path B processing...')
    return {'graph_state': state['graph_state'] + " simple graph Agents"}

print("✅ Conditional workflow nodes created!")
print("🔄 Flow: Node1 (common) → [Node2 OR Node3] (conditional paths)")

In [ ]:
# The decision-making brain of our conditional workflow
def decide_flow(state) -> Literal["call node_2", "call node_3"]:
    """
    Smart routing function that decides which path to take.
    
    This is where the magic happens - based on the current state,
    we can route to different specialized processing nodes.
    
    Args:
        state: Current workflow state containing graph_state
        
    Returns:
        String indicating which node to route to next
    """
    user_input = state['graph_state']  # Access current state
    
    # Random decision for demonstration (50/50 split)
    # In real applications, this could be based on:
    # - User input content analysis
    # - Previous processing results  
    # - External API calls
    # - Machine learning model predictions
    if random.random() > 0.5:
        print("🎯 Decision: Routing to Node 2 (building path)")
        return 'call node_2'
    else:
        print("🎯 Decision: Routing to Node 3 (agents path)")
        return 'call node_3'

print("✅ Decision function created!")
print("🧠 This function demonstrates intelligent routing in LangGraph")

In [ ]:
# Build the conditional workflow graph
from langgraph.graph import StateGraph, START, END

print("🏗️ Constructing conditional workflow with smart routing...")

# Initialize graph with our State schema
workflow = StateGraph(State)

# Add all processing nodes
workflow.add_node('node_1', node_1)  # Common initial processing
workflow.add_node('node_2', node_2)  # Specialized path A
workflow.add_node('node_3', node_3)  # Specialized path B

print("✅ Added all processing nodes")

# Define the workflow structure
workflow.add_edge(START, 'node_1')  # Everyone starts at node_1
print("🚀 Set entry point: node_1")

# This is the key: conditional routing based on decision logic
workflow.add_conditional_edges(
    'node_1',                    # From node_1...
    decide_flow,                 # Use this function to decide...
    {
        'call node_2': 'node_2',  # If returns 'call node_2', go to node_2
        'call node_3': 'node_3'   # If returns 'call node_3', go to node_3
    }
)
print("🔀 Added conditional routing from node_1")

# Both paths end the workflow
workflow.add_edge('node_2', END)  # Path A ends here
workflow.add_edge('node_3', END)  # Path B ends here
print("🏁 Set end points for both paths")

# Compile into executable agent
app = workflow.compile()

print("\n✅ Conditional workflow compiled successfully!")
print("📊 Structure: START → node_1 → [node_2 OR node_3] → END")
print("🎯 Each run will take a different path based on the decision logic!")

## 9. 🎨 Visualizing Conditional Logic

### See the Decision Tree in Action! 🌳

In [ ]:
# Visualize the conditional workflow
from IPython.display import Image, display

print("🎨 Generating visualization of conditional workflow...")
print("🔍 Notice the diamond shape showing the decision point!")

# Display the graph with conditional edges
display(Image(app.get_graph().draw_mermaid_png()))

print("✅ Conditional workflow diagram generated!")
print("📊 The diagram shows the branching logic at node_1")

## 10. 🧪 Testing Your Intelligent Agent

### Watch Different Paths in Action! 🎭
Let's run our conditional agent multiple times to see how it makes different decisions.

In [ ]:
# Test the conditional workflow with multiple runs
print("🧪 Testing conditional agent workflow...")
print("🎯 Watch how different runs take different paths!")
print("=" * 60)

# Run the graph multiple times to see different paths
for i in range(5):
    print(f"\n🔄 Run #{i+1}:")
    print("-" * 30)
    
    # Invoke the agent with initial state
    result = app.invoke({"graph_state": "Hi, my name is John. "})
    
    # Show the final result
    final_text = result['graph_state']
    print(f"📝 Final result: '{final_text}'")
    
    # Analyze which path was taken
    if " building" in final_text:
        print("🎯 Path taken: Node 1 → Node 2 (building path)")
    elif " simple graph Agents" in final_text:
        print("🎯 Path taken: Node 1 → Node 3 (agents path)")

print("\n" + "=" * 60)
print("✅ Testing complete! Notice how each run potentially takes a different path.")
print("🎭 This demonstrates the power of conditional logic in LangGraph!")

## 🎉 Congratulations! You've Built Your First Intelligent Agents!

### 🏆 What You've Accomplished

Amazing work! You've just completed a comprehensive journey through LangGraph fundamentals. Here's what you've mastered:

#### ✅ Core Skills Acquired:
1. **🔧 Environment Setup** - Configured LangGraph, LangChain, and Azure OpenAI
2. **📊 State Management** - Created TypedDict schemas for agent memory
3. **🏗️ Basic Workflows** - Built linear agent pipelines with multiple nodes
4. **🧠 Conditional Logic** - Implemented smart routing and decision-making
5. **🎨 Visualization** - Generated beautiful diagrams of your agent workflows
6. **🧪 Testing & Validation** - Ran comprehensive tests to verify functionality

#### 🚀 Key Concepts Mastered:
- **StateGraph**: The foundation of all LangGraph agents
- **Nodes**: Individual processing units in your workflow
- **Edges**: Connections that define data flow
- **Conditional Routing**: Dynamic path selection based on logic
- **State Persistence**: Maintaining context across processing steps

### 🌟 Real-World Applications

The patterns you've learned here are the building blocks for:
- **🤖 Chatbots & Virtual Assistants** - Multi-turn conversations with memory
- **📋 Workflow Automation** - Complex business process automation
- **🔍 Research Agents** - Multi-step information gathering and analysis
- **🎯 Decision Support Systems** - Intelligent routing based on context
- **📊 Data Processing Pipelines** - Multi-stage data transformation workflows

## 🚀 Next Steps: Your LangGraph Journey Continues

### 🎯 Immediate Next Steps

Ready to take your skills to the next level? Here's your roadmap:

#### 📚 **Continue Learning:**
1. **RAG Agents** - Build agents that can search and retrieve information
2. **Tool Integration** - Connect your agents to external APIs and services  
3. **Memory Systems** - Implement persistent memory across sessions
4. **Multi-Agent Systems** - Coordinate multiple specialized agents
5. **Production Deployment** - Scale your agents for real-world use

#### 🛠️ **Hands-On Practice:**
- **Experiment** with different decision logic in conditional edges
- **Modify** the state schemas to include more complex data
- **Create** agents for your specific use cases
- **Combine** multiple patterns into sophisticated workflows

### 📖 Resources for Continued Learning

#### 🌐 **Official Documentation:**
- [LangGraph Documentation](https://python.langchain.com/docs/langgraph)
- [LangChain Community Hub](https://github.com/langchain-ai/langchain)
- [Azure OpenAI Documentation](https://docs.microsoft.com/en-us/azure/cognitive-services/openai/)

#### 💡 **Advanced Patterns to Explore:**
- **Human-in-the-Loop Workflows** - Agents that request human input
- **Parallel Processing** - Multiple paths executing simultaneously  
- **Error Recovery** - Robust error handling and retry logic
- **Dynamic Graph Generation** - Graphs that modify themselves at runtime

### 🎯 **Challenge Yourself:**

Try building these progressively more complex agents:
1. **📝 Note-Taking Agent** - Summarizes and organizes information
2. **🔍 Research Assistant** - Gathers and synthesizes information from multiple sources
3. **📊 Data Analysis Agent** - Processes datasets and generates insights
4. **🎯 Recommendation Engine** - Makes personalized suggestions based on user preferences

---

### 🌟 **Final Words**

You've taken the first crucial steps into the world of agentic AI! LangGraph opens up endless possibilities for creating intelligent, adaptive systems. The patterns you've learned here - state management, conditional logic, and workflow design - are the foundation for building sophisticated AI applications.

**Keep experimenting, keep building, and most importantly, keep learning!** 🚀

---

*🎓 **Remember:** Every expert was once a beginner. You're now equipped with the fundamentals to build amazing AI agents. The future of AI development is in your hands!*